In [ ]:
# Standard library imports
import os
import tempfile

# Third-party imports
import pandas as pd
import requests

## Using the MPC Observations API

We make use of the [MPC Observations API](https://minorplanetcenter.net/mpcops/documentation/observations-api/) to obtain data on solar system objects in a variety of formats. These can be XML (shudder...), ADES or classic MPC1992 80 column format.

### Define URL for calling the MPC service

In [ ]:
query_url = 'https://data.minorplanetcenter.net/api/get-obs'

### Define target to query and run the query

In [ ]:
target = '2009 DP2'
response = requests.get(query_url, json={'desigs': [target], 'output_format': ['ADES_DF', 'OBS_DF']})

if response.ok:
    ades_df = pd.DataFrame(response.json()[0]['ADES_DF'])
    obs_df = pd.DataFrame(response.json()[0]['OBS_DF'])
    print(ades_df)
    # print(obs_df)
else:
    print('Error: ', response.status_code, response.content)

### Save to disk as CSV

In [ ]:
output_dir = tempfile.TemporaryDirectory(
    suffix='ades_',
    prefix='fomo_',
)
output_filename = os.path.join(output_dir.name, f'{target}_ADES.csv')
ades_df.to_csv(output_filename, index=False)
print(f'ADES data for {target} saved to: {output_filename}')

In [ ]:
from fomo.compare_utils import compare_ades_with_csv

# Compare the in-memory `ades_df` with the CSV we just wrote
res = compare_ades_with_csv(ades_df, output_filename)
print('Match:', res['match'])
if not res['match']:
    print(res.get('message'))
    # Safely print diffs without using DataFrame truthiness
    diff = res.get('diff')
    if diff is not None:
        try:
            print(diff.head(20))
        except Exception:
            print(diff)
    else:
        diff_rows = res.get('diff_rows')
        if diff_rows is not None:
            try:
                print(diff_rows.head(20))
            except Exception:
                print(diff_rows)
        else:
            print('No diff available')
else:
    print('hash_ades:', res.get('hash_ades'))
    print('hash_disk:', res.get('hash_disk'))